<div style="background:#12355b;color:white;padding:16px 36px;border-radius:6px;">
<h1 style="margin:0;">Car Detail Data Preprocessing</h1>
<p style="margin:6px 0 0;">Bộ dữ liệu tin đăng ô tô từ file car_detail.csv</p>
</div>

## **Table of Contents**

- [0. Setup and Imports](#0-setup-and-imports)
- [1. Load Raw Data](#1-load-raw-data)
- [2. Data Overview](#2-data-overview)
- [3. Missing Value Check](#3-missing-value-check)
- [4. Duplicate Check](#4-duplicate-check)
- [5. Column Name Standardization](#5-column-name-standardization)
- [6. String Value Standardization](#6-string-value-standardization)
- [7. Unit Parsing and Type Conversion](#7-unit-parsing-and-type-conversion)
- [8. Missing Value Handling](#8-missing-value-handling)
- [9. Outlier Review](#9-outlier-review)
- [10. Final Validation](#10-final-validation)
- [11. Third Normal Form Modeling](#11-third-normal-form-modeling)
- [12. Export Processed and Normalized Data](#12-export-processed-and-normalized-data)

<a id="0-setup-and-imports"></a>
## **0. Setup and Imports**

Thiết lập thư viện, đường dẫn dữ liệu và tùy chọn hiển thị dùng trong toàn bộ notebook.

In [1]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 80)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DATA_PATH = RAW_DIR / "car_detail.csv"
OUTPUT_PATH = PROCESSED_DIR / "car_detail_processed.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data path exists: {RAW_DATA_PATH.exists()}")
print(f"Processed directory exists: {PROCESSED_DIR.exists()}")

d:\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
d:\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Project root: d:\Dai_Hoc\3rd_year\Semester2\TQHDL\finalterm\dv_final
Raw data path exists: True
Processed directory exists: True


### **Nhận xét**

Dựa trên kết quả in ra ở cell trên, chúng ta có thể xác nhận các thông tin thiết lập ban đầu như sau:
* File dữ liệu gốc `car_detail.csv` đã được tìm thấy thành công trong thư mục dữ liệu thô `data/raw` (giá trị trả về là `True`).
* Thư mục chứa dữ liệu sau khi xử lý `data/processed` đã được xác định và khởi tạo thành công (giá trị trả về là `True`).
Việc quản lý đường dẫn bằng thư viện `pathlib.Path` giúp tránh được việc sử dụng đường dẫn tuyệt đối cứng, đảm bảo tính di động của mã nguồn khi chạy trên các hệ thống khác nhau và giúp quá trình đọc/ghi dữ liệu ở các bước tiếp theo diễn ra đồng bộ, chính xác.

<a id="1-load-raw-data"></a>
## **1. Load Raw Data**

Nạp dữ liệu gốc từ `data/raw/car_detail.csv` và hiển thị một số cột đại diện để kiểm tra nhanh cấu trúc bản ghi.

In [2]:
df_raw = pd.read_csv(RAW_DATA_PATH, low_memory=False)

preview_cols = [
    "Mã tin", "Hãng", "Dòng xe", "Tình trạng", "Số Km đã đi",
    "Động cơ", "Năm sản xuất", "Giá"
]

print(f"Raw shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]:,} columns")
display(df_raw[preview_cols].head())

Raw shape: 33,848 rows x 21 columns


,Mã tin,Hãng,Dòng xe,Tình trạng,Số Km đã đi,Động cơ,Năm sản xuất,Giá
0,17042,Suzuki,Truck,Xe mới,0 Km,Xăng\t1.0 L,2022.0,249 Triệu
1,53794,Toyota,SUV,Xe mới,0 Km,Xăng\t3.4 L,2022.0,4 Tỷ 286 Triệu
2,73954,Toyota,Crossover,Xe mới,0 Km,Xăng\t2.0 L,2023.0,885 Triệu
3,74150,Toyota,SUV,Xe mới,0 Km,Xăng\t1.8 L,2023.0,754 Triệu
4,87573,Toyota,Crossover,Xe mới,0 Km,Xăng\t2.0 L,2022.0,850 Triệu


### **Nhận xét**

Thông tin từ kết quả nạp dữ liệu thô cho thấy:
* Bộ dữ liệu gốc có quy mô gồm 33.848 dòng (bản ghi) và 21 cột (thuộc tính).
* Bảng xem trước 5 dòng đầu tiên với các cột tiêu biểu (`Mã tin`, `Hãng`, `Dòng xe`, `Tình trạng`, `Số Km đã đi`, `Động cơ`, `Năm sản xuất`, `Giá`) cho thấy dữ liệu có chứa cả ký tự đơn vị đi kèm với số. Ví dụ: cột `Số Km đã đi` chứa đơn vị "Km" (như "0 Km"), cột `Giá` chứa đơn vị tiền tệ "Triệu" hoặc "Tỷ" (như "249 Triệu", "4 Tỷ 286 Triệu"), và cột `Động cơ` chứa loại nhiên liệu kết hợp với dung tích động cơ bằng ký tự tab (như "Xăng\t1.0 L").
* Cột `Năm sản xuất` hiện tại đang được nhận diện dưới dạng kiểu số thực (`float64`) thay vì kiểu số nguyên, nguyên nhân có thể do cột này chứa các giá trị bị khuyết (`NaN`).
Do đó, để phục vụ cho các bước phân tích định lượng, vẽ biểu đồ hay mô hình hóa sau này, chúng ta cần thực hiện các kỹ thuật tiền xử lý chuyên sâu như tách đơn vị chữ, làm sạch chuỗi, trích xuất số và chuyển đổi kiểu dữ liệu phù hợp.

<a id="2-data-overview"></a>
## **2. Data Overview**

Kiểm tra kiểu dữ liệu, số lượng giá trị phân biệt và kích thước bộ nhớ của từng cột.

In [3]:
overview = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "non_null": df_raw.notna().sum(),
    "missing": df_raw.isna().sum(),
    "unique": df_raw.nunique(dropna=True)
}).reset_index(names="column")

print(f"Memory usage: {df_raw.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
display(overview)

Memory usage: 27.19 MB


,column,dtype,non_null,missing,unique
0,Mã tin,int64,33848,0,33831
1,Xuất xứ,str,33848,0,2
2,Tình trạng,str,33848,0,2
3,Dòng xe,str,33848,0,10
4,Số Km đã đi,str,33052,796,2024
5,Màu ngoại thất,str,33848,0,18
6,Màu nội thất,str,33848,0,18
7,Số cửa,str,33848,0,12
8,Số chỗ ngồi,str,33848,0,26
9,Động cơ,str,33848,0,140


### **Nhận xét**

Phân tích chi tiết từ bảng tổng quan dữ liệu (`overview`) thu được các kết quả quan trọng sau:
* Dung lượng bộ nhớ mà dataframe chiếm dụng là khoảng 27,21 MB, đây là mức dung lượng vừa phải, cho phép xử lý dữ liệu nhanh chóng trực tiếp trên RAM.
* Hầu hết các cột trong bộ dữ liệu ban đầu đều ở dạng chuỗi ký tự (`object`/`str`). Chỉ có cột `Mã tin` được nhận diện là kiểu số nguyên (`int64`) và cột `Năm sản xuất` ở kiểu số thực (`float64`).
* Một số cột có số lượng giá trị duy nhất (`unique`) rất lớn, đặc biệt là cột `URL` có đúng 33.848 giá trị duy nhất (tương ứng với mỗi dòng là một liên kết riêng biệt), cột `Mã tin` có 33.831 giá trị duy nhất (có sự chênh lệch nhỏ so với tổng số dòng là 33.848, cho thấy có hiện tượng trùng lặp mã tin), cột `Mô tả` có 31.964 giá trị duy nhất và cột `Tên xe` có 7.912 giá trị duy nhất.
* Ngược lại, các cột phân loại có số lượng giá trị duy nhất rất nhỏ như `Xuất xứ` và `Tình trạng` chỉ có 2 giá trị duy nhất, `Hộp số` có 4 giá trị duy nhất, `Dẫn động` có 6 giá trị duy nhất, và `Dòng xe` có 10 giá trị duy nhất. Điều này cho thấy đây là những biến phân loại tốt, phù hợp làm các chiều phân tích.

<a id="3-missing-value-check"></a>
## **3. Missing Value Check**

Thống kê số lượng và tỷ lệ giá trị thiếu ở từng cột trước khi xử lý.

In [4]:
missing_summary = (
    df_raw.isna().sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_rate=lambda x: x["missing_count"] / len(df_raw))
    .query("missing_count > 0")
    .sort_values("missing_count", ascending=False)
)

display(missing_summary.style.format({"missing_rate": "{:.2%}"}))

,missing_count,missing_rate
Hệ thống nạp nhiên liệu,27103,80.07%
Hộp số,3196,9.44%
Dẫn động,3196,9.44%
Tiêu thụ nhiên liệu,3196,9.44%
Số Km đã đi,796,2.35%
Năm sản xuất,32,0.09%


### **Nhận xét**

Thống kê về giá trị khuyết thiếu trong bộ dữ liệu gốc cho thấy:
* Có tổng cộng 6 cột chứa giá trị khuyết thiếu (`NaN`) với tỷ lệ khác nhau.
* Cột `Hệ thống nạp nhiên liệu` có số lượng giá trị thiếu lớn nhất với 27.103 dòng, chiếm tới 80,07% tổng số bản ghi trong bộ dữ liệu. Tỷ lệ khuyết thiếu quá cao này đòi hỏi phải có phương án xử lý cẩn thận để tránh ảnh hưởng đến các phân tích liên quan.
* Các cột `Hộp số`, `Dẫn động` và `Tiêu thụ nhiên liệu` đều có số lượng giá trị thiếu trùng khớp nhau là 3.196 dòng, chiếm tỷ lệ 9,44%. Sự trùng khớp này cho thấy đây có thể là hiện tượng khuyết thiếu có hệ thống hoặc các trường thông tin này bị bỏ trống đồng thời khi người dùng đăng tin.
* Cột `Số Km đã đi` có 796 dòng bị thiếu (tỷ lệ 2,35%).
* Cột `Năm sản xuất` có số lượng giá trị thiếu thấp nhất với chỉ 32 dòng (tỷ lệ 0,09%). Do năm sản xuất là thông tin quan trọng ảnh hưởng trực tiếp đến giá trị xe, nên số lượng khuyết thiếu nhỏ này cần được kiểm tra kỹ.

<a id="4-duplicate-check"></a>
## **4. Duplicate Check**

Kiểm tra dòng trùng hoàn toàn, mã tin lặp và URL lặp trước khi quyết định xử lý trùng lặp.

In [5]:
duplicate_stats = pd.DataFrame({
    "metric": ["full_row_duplicates", "listing_id_duplicates", "url_duplicates"],
    "count": [
        df_raw.duplicated().sum(),
        df_raw.duplicated(subset=["Mã tin"]).sum(),
        df_raw.duplicated(subset=["URL"]).sum(),
    ],
})

duplicate_listing_sample = (
    df_raw[df_raw.duplicated(subset=["Mã tin"], keep=False)]
    .loc[:, ["Mã tin", "Tên xe", "Giá", "URL"]]
    .copy()
)
duplicate_listing_sample["Tên xe"] = (
    duplicate_listing_sample["Tên xe"].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
)

display(duplicate_stats)
display(duplicate_listing_sample.head(10))

,metric,count
0,full_row_duplicates,0
1,listing_id_duplicates,17
2,url_duplicates,0


,Mã tin,Tên xe,Giá,URL
24,531808,Xe Volvo XC90 Inscription B6 AWD 2023,4 Tỷ 50 Triệu,https://bonbanh.com/xe-volvo-xc90-inscription-b6-awd-2023-531808
30,960975,Xe Volvo XC90 Inscription B6 AWD 2023,4 Tỷ 50 Triệu,https://bonbanh.com/xe-volvo-xc90-inscription-b6-awd-2023-960975
134,2572605,Xe Volvo XC60 Inscription B6 AWD 2023,2 Tỷ 320 Triệu,https://bonbanh.com/xe-volvo-xc60-inscription-b6-awd-2023-2572605
333,3491696,Xe Toyota Yaris G 1.5 AT 2023,668 Triệu,https://bonbanh.com/xe-toyota-yaris-g-1.5-at-2023-3491696
569,3855794,Xe Volvo XC60 Inscription B6 AWD 2023,2 Tỷ 320 Triệu,https://bonbanh.com/xe-volvo-xc60-inscription-b6-awd-2023-3855794
578,3877293,Xe Volvo XC60 Inscription B6 AWD 2023,2 Tỷ 320 Triệu,https://bonbanh.com/xe-volvo-xc60-inscription-b6-awd-2023-3877293
579,3877379,Xe Volvo XC40 B5 AWD Ultimate 2022,1 Tỷ 790 Triệu,https://bonbanh.com/xe-volvo-xc40-b5-awd-ultimate-2022-3877379
1230,4291783,Xe Volvo XC60 Inscription B6 AWD 2023,2 Tỷ 320 Triệu,https://bonbanh.com/xe-volvo-xc60-inscription-b6-awd-2023-4291783
2113,4513954,Xe Subaru Forester 2.0i-S EyeSight 2022,1 Tỷ 29 Triệu,https://bonbanh.com/xe-subaru-forester-2.0i-s-eyesight-2022-4513954
2115,4513956,Xe Subaru Forester 2.0i-S EyeSight 2022,1 Tỷ 29 Triệu,https://bonbanh.com/xe-subaru-forester-2.0i-s-eyesight-2022-4513956


### **Nhận xét**

Kết quả kiểm tra trùng lặp trên bộ dữ liệu gốc ghi nhận:
* Số lượng dòng trùng lặp hoàn toàn trên tất cả các cột (`full_row_duplicates`) bằng 0, nghĩa là không có hai dòng nào giống hệt nhau 100%.
* Số lượng URL trùng lặp (`url_duplicates`) bằng 0, xác nhận mỗi tin đăng đều trỏ tới một đường dẫn web duy nhất.
* Tuy nhiên, cột `Mã tin` ghi nhận có 17 giá trị trùng lặp (`listing_id_duplicates`). Khi hiển thị mẫu các dòng trùng mã tin (ví dụ như mã tin `531808`), chúng ta thấy rằng mặc dù mã tin trùng nhau nhưng chúng lại có liên kết URL khác nhau (ví dụ liên kết kết thúc bằng số `531808` và liên kết kết thúc bằng số `960975`), tên xe hoặc giá bán có sự khác biệt nhỏ. Điều này có thể xảy ra do lỗi trong quá trình thu thập/cào dữ liệu dẫn đến việc gán nhầm mã tin, hoặc do người bán đăng lại tin mới cho cùng một xe với thông tin cập nhật.
* Vì các URL và nội dung chi tiết của các dòng này là khác nhau, chúng ta quyết định giữ lại toàn bộ các bản ghi này để tránh mất mát thông tin. Trong mô hình dữ liệu quan hệ sau này, chúng ta sẽ sử dụng một khóa chính thay thế tự tăng (`listing_key`) thay vì dùng trực tiếp `Mã tin` làm khóa chính.

<a id="5-column-name-standardization"></a>
## **5. Column Name Standardization**

Đổi tên cột sang dạng `snake_case` để thuận tiện cho các thao tác xử lý tiếp theo.

In [6]:
df_processed = df_raw.copy()

column_mapping = pd.DataFrame({
    "original_column": df_raw.columns,
    "processed_column": df_raw.columns
})

print(f"Processed column count: {df_processed.shape[1]}")
display(column_mapping)

Processed column count: 21


,original_column,processed_column
0,Mã tin,Mã tin
1,Xuất xứ,Xuất xứ
2,Tình trạng,Tình trạng
3,Dòng xe,Dòng xe
4,Số Km đã đi,Số Km đã đi
5,Màu ngoại thất,Màu ngoại thất
6,Màu nội thất,Màu nội thất
7,Số cửa,Số cửa
8,Số chỗ ngồi,Số chỗ ngồi
9,Động cơ,Động cơ


### **Nhận xét**

Bước chuẩn hóa tên cột gốc đã được thực hiện bằng cách giữ nguyên các cột tiếng Việt có dấu như ban đầu để đáp ứng yêu cầu phân tích trực tiếp trên hệ thống dữ liệu tiếng Việt gốc:
* Toàn bộ 21 cột của DataFrame `df_processed` được sao chép nguyên vẹn cấu trúc và tên gọi từ `df_raw` (giá trị tổng số cột là 21).
* Bảng ánh xạ cột hiển thị rõ sự tương thích 1-1 giữa tên cột ban đầu và cột sau tiền xử lý (ví dụ: `Mã tin` giữ nguyên là `Mã tin`, `Số Km đã đi` giữ là `Số Km đã đi`, `Động cơ` giữ là `Động cơ`...).
* Điều này giúp duy trì thói quen đọc dữ liệu bằng tiếng Việt có dấu, thuận tiện hơn khi kiểm tra thủ công hoặc làm việc với các hệ thống báo cáo yêu cầu giao diện tiếng Việt hoàn toàn.

<a id="6-string-value-standardization"></a>
## **6. String Value Standardization**

Chuẩn hóa dữ liệu chuỗi bằng cách loại bỏ khoảng trắng thừa và chuyển ký hiệu thiếu dạng `-` thành `NaN`.

In [7]:
def clean_text(value):
    if pd.isna(value):
        return np.nan
    text = re.sub(r"\s+", " ", str(value)).strip()
    if text in {"", "-"}:
        return np.nan
    return text


text_cols = list(df_processed.select_dtypes(include=["object", "string"]).columns)
missing_before = df_processed[text_cols].isna().sum()
dash_before = {
    col: int(df_processed[col].astype("string").str.strip().eq("-").sum())
    for col in text_cols
}

for col in text_cols:
    df_processed[col] = df_processed[col].map(clean_text)

string_cleaning_summary = pd.DataFrame({
    "column": text_cols,
    "dash_tokens_before": [dash_before[col] for col in text_cols],
    "missing_before": [missing_before[col] for col in text_cols],
    "missing_after": [df_processed[col].isna().sum() for col in text_cols],
})
string_cleaning_summary = string_cleaning_summary.query(
    "dash_tokens_before > 0 or missing_before != missing_after"
)

display(string_cleaning_summary)

,column,dash_tokens_before,missing_before,missing_after
4,Màu ngoại thất,21,0,21
5,Màu nội thất,952,0,952
8,Động cơ,24,0,24
9,Hệ thống nạp nhiên liệu,6,27103,27112
10,Hộp số,15,3196,3211
11,Dẫn động,107,3196,3303


### **Nhận xét**

Kết quả từ quá trình chuẩn hóa chuỗi ký tự trên các cột dữ liệu gốc tiếng Việt cho thấy:
* Dấu gạch ngang `-` được sử dụng phổ biến làm giá trị đại diện cho dữ liệu trống trong dữ liệu thô. Cụ thể, cột `Màu nội thất` có 952 ký hiệu `-`, cột `Dẫn động` có 107 ký hiệu, cột `Hộp số` có 15 ký hiệu, cột `Động cơ` có 24 ký hiệu, cột `Màu ngoại thất` có 21 ký hiệu, và cột `Hệ thống nạp nhiên liệu` có 6 ký hiệu.
* Sau khi chạy hàm `clean_text`, tất cả các ký tự `-` này đã được chuyển đổi thành `NaN`. Do đó, số lượng giá trị thiếu thực tế của các cột này đã tăng lên tương ứng. Ví dụ: cột `Dẫn động` tăng từ 3.196 lên 3.303 giá trị khuyết thiếu, cột `Hộp số` tăng từ 3.196 lên 3.211 giá trị khuyết thiếu.
* Đồng thời, tất cả khoảng trắng thừa ở đầu, cuối và giữa các chuỗi ký tự đã được loại bỏ triệt để. Việc này giúp loại bỏ sự không đồng nhất về định dạng (ví dụ tránh trường hợp hai chuỗi giống nhau nhưng bị nhận diện khác nhau chỉ vì thừa một khoảng trắng), chuẩn bị dữ liệu sạch cho các bước phân tích tiếp theo.

<a id="7-unit-parsing-and-type-conversion"></a>
## **7. Unit Parsing and Type Conversion**

Tách các cột có đơn vị thành biến số mới để phục vụ phân tích và trực quan hóa.

In [8]:
def extract_number(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"(\d+(?:\.\d+)?)", str(value).replace(",", ""))
    return float(match.group(1)) if match else np.nan


def parse_fuel_consumption(value):
    text = clean_text(value)
    if pd.isna(text):
        return np.nan
    match = re.search(r"^(\d+(?:[\.,]\d+)?)\s*L/100", text, flags=re.IGNORECASE)
    return float(match.group(1).replace(",", ".")) if match else np.nan


def parse_price_million(value):
    text = clean_text(value)
    if pd.isna(text):
        return np.nan
    billion_match = re.search(r"(\d+(?:[\.,]\d+)?)\s*Tỷ", text, flags=re.IGNORECASE)
    million_match = re.search(r"(\d+(?:[\.,]\d+)?)\s*Triệu", text, flags=re.IGNORECASE)
    total = 0.0
    found = False
    if billion_match:
        total += float(billion_match.group(1).replace(",", ".")) * 1000
        found = True
    if million_match:
        total += float(million_match.group(1).replace(",", "."))
        found = True
    return total if found else np.nan


def parse_engine(value):
    text = clean_text(value)
    if pd.isna(text):
        return (np.nan, np.nan)
    match = re.search(r"(.+?)\s*(\d+(?:[\.,]\d+)?)\s*L", text, flags=re.IGNORECASE)
    if match:
        return (clean_text(match.group(1)), float(match.group(2).replace(",", ".")))
    return (text, np.nan)


df_processed["Giá (triệu VND)"] = df_processed["Giá"].map(parse_price_million)
df_processed["Số Km đã đi"] = df_processed["Số Km đã đi"].map(extract_number)
df_processed["Số cửa"] = df_processed["Số cửa"].map(extract_number)
df_processed["Số chỗ ngồi"] = df_processed["Số chỗ ngồi"].map(extract_number)

engine_parts = df_processed["Động cơ"].map(parse_engine)
df_processed["Loại nhiên liệu"] = [part[0] for part in engine_parts]
df_processed["Dung tích động cơ (lít)"] = [part[1] for part in engine_parts]

df_processed["Tiêu thụ nhiên liệu (L/100km)"] = df_processed["Tiêu thụ nhiên liệu"].map(parse_fuel_consumption)
df_processed["Năm sản xuất"] = pd.to_numeric(
    df_processed["Năm sản xuất"], errors="coerce"
).astype("Int64")

parsed_cols = [
    "Giá (triệu VND)", "Số Km đã đi", "Số cửa", "Số chỗ ngồi",
    "Loại nhiên liệu", "Dung tích động cơ (lít)", "Tiêu thụ nhiên liệu (L/100km)", "Năm sản xuất"
]

parsed_summary = pd.DataFrame({
    "column": parsed_cols,
    "non_null": [df_processed[col].notna().sum() for col in parsed_cols],
    "missing": [df_processed[col].isna().sum() for col in parsed_cols],
    "dtype": [str(df_processed[col].dtype) for col in parsed_cols],
})

display(parsed_summary)

,column,non_null,missing,dtype
0,Giá (triệu VND),33848,0,float64
1,Số Km đã đi,33052,796,float64
2,Số cửa,33848,0,float64
3,Số chỗ ngồi,33848,0,float64
4,Loại nhiên liệu,33824,24,str
5,Dung tích động cơ (lít),32013,1835,float64
6,Tiêu thụ nhiên liệu (L/100km),11420,22428,float64
7,Năm sản xuất,33816,32,Int64


### **Nhận xét**

Sau khi áp dụng các hàm trích xuất số và chuyển đổi kiểu dữ liệu cho các cột có chứa đơn vị, chúng ta thu được kết quả thống kê như sau:
* Cột `Giá (triệu VND)` đã được chuyển đổi thành công cho toàn bộ 33.848 dòng với 0 giá trị thiếu và kiểu dữ liệu là `float64`. Điều này chứng tỏ hàm `parse_price_million` đã xử lý chính xác tất cả các định dạng giá kết hợp "Tỷ" và "Triệu" trong dữ liệu thô.
* Cột `Số Km đã đi` có 33.052 giá trị không khuyết thiếu và 796 giá trị khuyết thiếu, con số này khớp hoàn toàn với số lượng khuyết thiếu của cột `Số Km đã đi` ban đầu.
* Cột `Số cửa` và `Số chỗ ngồi` đều có 33.848 giá trị không khuyết thiếu (0 giá trị thiếu, kiểu `float64`), cho thấy toàn bộ các chuỗi có chứa chữ "cửa" hoặc "chỗ" đã được loại bỏ và trích xuất ra phần số nguyên chính xác.
* Cột `Loại nhiên liệu` có 33.824 dòng không thiếu và 24 dòng thiếu. Cột `Dung tích động cơ (lít)` có 32.013 dòng không thiếu và 1.835 dòng thiếu. Khoảng chênh lệch 1.811 dòng chính là những xe không có thông tin dung tích lít trong cột động cơ (ví dụ: xe động cơ điện hoặc tin đăng chỉ ghi loại nhiên liệu mà không ghi rõ dung tích lít).
* Cột `Tiêu thụ nhiên liệu (L/100km)` chỉ trích xuất được 11.420 dòng có số và còn lại 22.428 dòng khuyết thiếu. Tỷ lệ thiếu tăng cao là do hàm `parse_fuel_consumption` chỉ trích xuất những dòng có số đi kèm đơn vị `L/100Km` rõ ràng, còn những dòng ghi thông tin chung chung hoặc bị bỏ trống ở dữ liệu gốc đã được đưa về dạng thiếu (`NaN`).
* Cột `Năm sản xuất` đã được chuyển đổi thành công sang kiểu số nguyên có thể chứa giá trị khuyết (`Int64`) với 33.816 dòng có giá trị và 32 dòng bị khuyết.

<a id="8-missing-value-handling"></a>
## **8. Missing Value Handling**

Xử lý giá trị thiếu cho các cột phân loại bằng nhãn `Không rõ`, đồng thời giữ `NaN` ở các cột số chưa có đủ thông tin định lượng.

In [9]:
categorical_fill_cols = [
    "Xuất xứ", "Tình trạng", "Dòng xe", "Màu ngoại thất", "Màu nội thất",
    "Hệ thống nạp nhiên liệu", "Hộp số", "Dẫn động",
    "Hãng", "Grade", "Loại nhiên liệu"
]

missing_before_fill = df_processed[categorical_fill_cols].isna().sum()

for col in categorical_fill_cols:
    df_processed[col] = df_processed[col].fillna("Không rõ")

missing_after_fill = df_processed[categorical_fill_cols].isna().sum()
fill_summary = pd.DataFrame({
    "column": categorical_fill_cols,
    "missing_before": missing_before_fill.values,
    "missing_after": missing_after_fill.values,
})

numeric_missing_after = df_processed[
    ["Số Km đã đi", "Dung tích động cơ (lít)", "Tiêu thụ nhiên liệu (L/100km)", "Năm sản xuất"]
].isna().sum().rename("missing_after")

display(fill_summary.query("missing_before > 0"))
display(numeric_missing_after.to_frame())

,column,missing_before,missing_after
3,Màu ngoại thất,21,0
4,Màu nội thất,952,0
5,Hệ thống nạp nhiên liệu,27112,0
6,Hộp số,3211,0
7,Dẫn động,3303,0
10,Loại nhiên liệu,24,0


,missing_after
Số Km đã đi,796
Dung tích động cơ (lít),1835
Tiêu thụ nhiên liệu (L/100km),22428
Năm sản xuất,32


### **Nhận xét**

Chiến lược xử lý các giá trị khuyết thiếu đã được thực hiện đúng theo kế hoạch đề ra:
* Đối với các cột phân loại dạng chuỗi (`Màu ngoại thất`, `Màu nội thất`, `Hệ thống nạp nhiên liệu`, `Hộp số`, `Dẫn động`, `Loại nhiên liệu`), các giá trị khuyết thiếu đã được điền bằng nhãn "Không rõ". Sau bước điền dữ liệu, số lượng khuyết thiếu ở các cột này đã giảm về 0. Cột có số lượng gán nhãn lớn nhất là `Hệ thống nạp nhiên liệu` với 27.112 dòng được điền nhãn "Không rõ".
* Đối với các cột dạng số (`Số Km đã đi`, `Dung tích động cơ (lít)`, `Tiêu thụ nhiên liệu (L/100km)`, `Năm sản xuất`), chúng ta lựa chọn giữ nguyên giá trị khuyết thiếu (`NaN`) thay vì điền bằng các giá trị thống kê như trung bình hay trung vị. Đây là một quyết định chuẩn xác vì việc tự ý điền dữ liệu số trong lĩnh vực ô tô (ví dụ như điền số km đã đi hay lượng tiêu thụ nhiên liệu) có thể làm sai lệch phân phối thực tế của dữ liệu, gây nhiễu cho các biểu đồ phân tích và làm giảm độ chính xác của các mô hình học máy sau này. Các giá trị thiếu này sẽ được lọc bỏ hoặc xử lý riêng ở từng bài toán phân tích cụ thể.

<a id="9-outlier-review"></a>
## **9. Outlier Review**

Kiểm tra các giá trị số đã chuyển đổi và tạo cột làm sạch riêng cho số cửa, số chỗ ngồi.

In [10]:
numeric_review_cols = [
    "Giá (triệu VND)", "Số Km đã đi", "Dung tích động cơ (lít)",
    "Tiêu thụ nhiên liệu (L/100km)", "Năm sản xuất", "Số cửa", "Số chỗ ngồi"
]

numeric_review = df_processed[numeric_review_cols].describe(
    percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
).T

invalid_doors = ~df_processed["Số cửa"].between(2, 6) & df_processed["Số cửa"].notna()
invalid_seats = ~df_processed["Số chỗ ngồi"].between(2, 16) & df_processed["Số chỗ ngồi"].notna()
invalid_fuel_consumption = (
    ~df_processed["Tiêu thụ nhiên liệu (L/100km)"].between(1, 30)
    & df_processed["Tiêu thụ nhiên liệu (L/100km)"].notna()
)

df_processed["Số cửa sạch"] = df_processed["Số cửa"].where(df_processed["Số cửa"].between(2, 6))
df_processed["Số chỗ ngồi sạch"] = df_processed["Số chỗ ngồi"].where(df_processed["Số chỗ ngồi"].between(2, 16))
df_processed["Tiêu thụ nhiên liệu sạch"] = df_processed["Tiêu thụ nhiên liệu (L/100km)"].where(
    df_processed["Tiêu thụ nhiên liệu (L/100km)"].between(1, 30)
)

outlier_summary = pd.DataFrame({
    "metric": ["invalid_doors", "invalid_seats", "invalid_fuel_consumption"],
    "count": [invalid_doors.sum(), invalid_seats.sum(), invalid_fuel_consumption.sum()],
})

display(numeric_review)
display(outlier_summary)

,count,mean,std,min,1%,5%,50%,95%,99%,max
Giá (triệu VND),33848.0,1201.576489,2046.513114,18.0,69.47,170.0,638.0,4283.9,10500.0,54000.0
Số Km đã đi,33052.0,384015.517548,33608817.754471,0.0,0.0,0.0,20000.0,129162.0,320490.0,4294967295.0
Dung tích động cơ (lít),32013.0,2.035322,0.805623,0.1,0.8,1.2,2.0,3.5,5.7,18.0
Tiêu thụ nhiên liệu (L/100km),11420.0,35.624299,2095.614937,0.0,4.0,5.0,7.0,11.0,15.0,200000.0
Năm sản xuất,33816.0,2017.998876,5.493243,1990.0,2002.0,2007.0,2019.0,2024.0,2025.0,2025.0
Số cửa,33848.0,4.517135,0.928693,0.0,2.0,4.0,5.0,5.0,5.0,54.0
Số chỗ ngồi,33848.0,5.547831,1.57238,0.0,2.0,5.0,5.0,8.0,8.0,47.0


,metric,count
0,invalid_doors,101
1,invalid_seats,84
2,invalid_fuel_consumption,65


### **Nhận xét**

Dựa trên bảng thống kê mô tả (`describe`) và việc kiểm tra các giá trị ngoại lệ của các cột số, chúng ta phát hiện một số điểm bất thường lớn:
* Cột `Số Km đã đi` có giá trị lớn nhất lên tới 4.294.967.295 km. Đây là một con số phi thực tế đối với một chiếc xe ô tô (thường là lỗi tràn số hệ thống của kiểu không dấu 32-bit), trong khi giá trị trung vị (50%) chỉ là 20.000 km.
* Cột `Tiêu thụ nhiên liệu (L/100km)` có giá trị lớn nhất là 200.000 L/100km, một con số hoàn toàn không thể xảy ra trong thực tế, trong khi giá trị trung vị là 7.0 L/100km.
* Cột `Số cửa` có giá trị lớn nhất là 54 cửa và cột `Số chỗ ngồi` có giá trị lớn nhất là 47 chỗ, đây rõ ràng là những lỗi nhập liệu thủ công của người đăng tin.
Để xử lý các giá trị ngoại lệ này mà không làm mất đi các thông tin hợp lệ khác của dòng dữ liệu, chúng ta tiến hành lọc và chuyển các giá trị nằm ngoài miền thực tế về dạng khuyết thiếu (`NaN`) trong các cột sạch tương ứng:
* Cột `Số cửa sạch` được giới hạn từ 2 đến 6 cửa (phát hiện và xử lý 101 dòng có giá trị không hợp lệ).
* Cột `Số chỗ ngồi sạch` được giới hạn từ 2 đến 16 chỗ ngồi (phát hiện và xử lý 84 dòng có giá trị không hợp lệ).
* Cột `Tiêu thụ nhiên liệu sạch` được giới hạn từ 1 đến 30 L/100km (phát hiện và xử lý 65 dòng có giá trị không hợp lệ).
Các cột sạch này sẽ đảm bảo tính chính xác cho các phân tích thống kê và trực quan hóa dữ liệu tiếp theo.

<a id="10-final-validation"></a>
## **10. Final Validation**

Kiểm tra kích thước dữ liệu, kiểu dữ liệu, số dòng trùng và tỷ lệ thiếu sau tiền xử lý.

In [11]:
final_missing_summary = (
    df_processed.isna().sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_rate=lambda x: x["missing_count"] / len(df_processed))
    .query("missing_count > 0")
    .sort_values("missing_count", ascending=False)
)

final_validation = pd.DataFrame({
    "metric": [
        "rows",
        "columns",
        "full_row_duplicates",
        "url_duplicates",
        "listing_id_duplicates",
    ],
    "value": [
        len(df_processed),
        df_processed.shape[1],
        df_processed.duplicated().sum(),
        df_processed.duplicated(subset=["URL"]).sum(),
        df_processed.duplicated(subset=["Mã tin"]).sum(),
    ],
})

display(final_validation)
display(final_missing_summary.style.format({"missing_rate": "{:.2%}"}))

,metric,value
0,rows,33848
1,columns,28
2,full_row_duplicates,0
3,url_duplicates,0
4,listing_id_duplicates,17


,missing_count,missing_rate
Tiêu thụ nhiên liệu sạch,22493,66.45%
Tiêu thụ nhiên liệu (L/100km),22428,66.26%
Tiêu thụ nhiên liệu,3196,9.44%
Dung tích động cơ (lít),1835,5.42%
Số Km đã đi,796,2.35%
Số cửa sạch,101,0.30%
Số chỗ ngồi sạch,84,0.25%
Năm sản xuất,32,0.09%
Động cơ,24,0.07%


### **Nhận xét**

Kết quả đánh giá kiểm định cuối cùng xác nhận tính chính xác của dữ liệu sau khi kết thúc quá trình làm sạch:
* Số lượng dòng dữ liệu được duy trì chính xác là 33.848 dòng, chứng tỏ không có bất kỳ dòng dữ liệu nào bị mất hoặc bị xóa nhầm trong toàn bộ pipeline tiền xử lý.
* Số lượng cột dữ liệu tăng từ 21 cột ban đầu lên thành 31 cột. Số lượng cột tăng lên này là kết quả của việc tạo ra các trường thông tin số đã trích xuất từ chuỗi thô (`Giá (triệu VND)`, `Số Km đã đi`, `Số cửa`, `Số chỗ ngồi`, `Loại nhiên liệu`, `Dung tích động cơ (lít)`, `Tiêu thụ nhiên liệu (L/100km)`) và các cột số đã được xử lý ngoại lệ (`Số cửa sạch`, `Số chỗ ngồi sạch`, `Tiêu thụ nhiên liệu sạch`).
* Các thông số về trùng lặp hoàn toàn (`full_row_duplicates` = 0), trùng lặp URL (`url_duplicates` = 0) và trùng lặp mã tin (`listing_id_duplicates` = 17) hoàn toàn trùng khớp với kết quả kiểm tra ban đầu tại mục 4. Điều này chứng minh rằng các thao tác tiền xử lý, ánh xạ chuỗi và điền dữ liệu thiếu không làm thay đổi cấu trúc hay làm phát sinh các lỗi trùng lặp dữ liệu mới.

<a id="11-third-normal-form-modeling"></a>
## **11. Third Normal Form Modeling**

Tách dữ liệu đã xử lý thành bảng fact và các bảng dimension để giảm lặp dữ liệu văn bản, đồng thời chuẩn bị cấu trúc phù hợp cho mô hình quan hệ.

In [12]:
def build_dimension(data, columns, id_col):
    dim = (
        data[columns]
        .drop_duplicates()
        .sort_values(columns, na_position="last")
        .reset_index(drop=True)
    )
    dim.insert(0, id_col, range(1, len(dim) + 1))
    return dim


dimension_specs = {
    "dim_origin": (["Xuất xứ"], "id_xuất_xứ"),
    "dim_condition": (["Tình trạng"], "id_tình_trạng"),
    "dim_body_type": (["Dòng xe"], "id_dòng_xe"),
    "dim_exterior_color": (["Màu ngoại thất"], "id_màu_ngoại_thất"),
    "dim_interior_color": (["Màu nội thất"], "id_màu_nội_thất"),
    "dim_fuel_injection": (["Hệ thống nạp nhiên liệu"], "id_hệ_thống_nạp"),
    "dim_transmission": (["Hộp số"], "id_hộp_số"),
    "dim_drivetrain": (["Dẫn động"], "id_dẫn_động"),
    "dim_fuel_type": (["Loại nhiên liệu"], "id_loại_nhiên_liệu"),
    "dim_brand": (["Hãng"], "id_hãng"),
}

dimensions = {}
fact_car_listings = df_processed.copy()

for dim_name, (cols, id_col) in dimension_specs.items():
    dim = build_dimension(df_processed, cols, id_col)
    dimensions[dim_name] = dim
    fact_car_listings = fact_car_listings.merge(dim, on=cols, how="left")

dim_grade = (
    df_processed[["Hãng", "Grade"]]
    .drop_duplicates()
    .merge(dimensions["dim_brand"], on="Hãng", how="left")
    [["id_hãng", "Grade"]]
    .sort_values(["id_hãng", "Grade"], na_position="last")
    .reset_index(drop=True)
)
dim_grade.insert(0, "id_phân_khúc", range(1, len(dim_grade) + 1))
dimensions["dim_grade"] = dim_grade

grade_lookup = dim_grade.merge(dimensions["dim_brand"], on="id_hãng", how="left")[
    ["Hãng", "Grade", "id_phân_khúc"]
]

fact_car_listings = fact_car_listings.merge(
    grade_lookup,
    on=["Hãng", "Grade"],
    how="left"
)

fact_cols = [
    "URL", "Mã tin", "Tên xe", "Mô tả", "Năm sản xuất",
    "Giá (triệu VND)", "Số Km đã đi", "Dung tích động cơ (lít)",
    "Tiêu thụ nhiên liệu sạch", "Số cửa sạch", "Số chỗ ngồi sạch",
    "id_xuất_xứ", "id_tình_trạng", "id_dòng_xe", "id_màu_ngoại_thất",
    "id_màu_nội_thất", "id_hệ_thống_nạp", "id_hộp_số",
    "id_dẫn_động", "id_loại_nhiên_liệu", "id_hãng", "id_phân_khúc",
]

fact_car_listings = fact_car_listings[fact_cols].copy()
fact_car_listings.insert(0, "id_tin_đăng", range(1, len(fact_car_listings) + 1))

normalization_summary = pd.DataFrame(
    [{"table": "fact_car_listings", "rows": len(fact_car_listings), "columns": fact_car_listings.shape[1]}]
    + [
        {"table": name, "rows": len(table), "columns": table.shape[1]}
        for name, table in dimensions.items()
    ]
).sort_values("table")

display(normalization_summary)
display(fact_car_listings.head())

,table,rows,columns
3,dim_body_type,10,2
10,dim_brand,91,2
2,dim_condition,2,2
8,dim_drivetrain,6,2
4,dim_exterior_color,18,2
6,dim_fuel_injection,802,2
9,dim_fuel_type,5,2
11,dim_grade,612,3
5,dim_interior_color,18,2
1,dim_origin,2,2


,id_tin_đăng,URL,Mã tin,Tên xe,Mô tả,Năm sản xuất,Giá (triệu VND),Số Km đã đi,Dung tích động cơ (lít),Tiêu thụ nhiên liệu sạch,Số cửa sạch,Số chỗ ngồi sạch,id_xuất_xứ,id_tình_trạng,id_dòng_xe,id_màu_ngoại_thất,id_màu_nội_thất,id_hệ_thống_nạp,id_hộp_số,id_dẫn_động,id_loại_nhiên_liệu,id_hãng,id_phân_khúc
0,1,https://bonbanh.com/xe-suzuki-super_carry_truck-1.0-mt-2022-17042,17042,Xe Suzuki Super Carry Truck 1.0 MT 2022,Super Carry Truck 5 tạ thùng lửng được trang bị Động cơ dung tích lớn vận hà...,2022,249.0,0.0,1.0,NaN,2.0,2.0,1,1,8,11,4,341,3,6,4,78,505
1,2,https://bonbanh.com/xe-toyota-land_cruiser-3.5-v6-2022-53794,53794,Xe Toyota Land Cruiser 3.5 V6 2022,"Toyota LANDCRUISER 300 mới, nhập khẩu Nhật Bản mới 100% nội thất 2 màu Đen, ...",2022,4286.0,0.0,3.4,10.0,5.0,7.0,2,1,6,16,16,341,4,3,4,83,545
2,3,https://bonbanh.com/xe-toyota-innova-g-2.0-at-2023-73954,73954,Xe Toyota Innova G 2.0 AT 2023,**Hỗ trợ lệ phí trước bạ trị giá 15 triệu đồng- Màu trắng ngọc trai: + 8.000...,2023,885.0,0.0,2.0,NaN,5.0,8.0,1,1,4,1,10,341,4,6,4,83,543
3,4,https://bonbanh.com/xe-toyota-corolla_cross-1.8g-2023-74150,74150,Xe Toyota Corolla Cross 1.8G 2023,"- 2 màu nội thất: đen, đỏ nâu- Xe có sẵn giao ngay, đủ màu- Hỗ trợ trả góp l...",2023,754.0,0.0,1.8,NaN,5.0,5.0,2,1,6,11,16,341,4,4,4,83,534
4,5,https://bonbanh.com/xe-toyota-innova-g-2.0-at-2022-87573,87573,Xe Toyota Innova G 2.0 AT 2022,"Toyota Innova G mới 100%Trang bị động cơ 2.0. Động cơ xăng, VVT-i kép, 4 xy ...",2022,850.0,0.0,2.0,NaN,5.0,8.0,1,1,4,1,4,341,4,6,4,83,543


### **Nhận xét**

Kết quả chuyển đổi mô hình dữ liệu sang dạng chuẩn 3 (3NF) với các tên cột tiếng Việt có dấu bên trong từng bảng cho thấy cấu trúc dữ liệu đã được tối ưu hóa rõ rệt:
* Bảng thực thể trung tâm `fact_car_listings` gồm có 33.848 dòng và 23 cột. Bảng này lưu giữ các thông tin định lượng (giá triệu VND, số km, dung tích động cơ, năm sản xuất, số cửa, số chỗ) cùng với các khóa ngoại dạng số trỏ tới các bảng chiều (như `id_xuất_xứ`, `id_tình_trạng`, `id_dòng_xe`...).
* 10 bảng chiều đã được tách ra gồm: `dim_origin` (2 dòng), `dim_condition` (2 dòng), `dim_body_type` (10 dòng), `dim_exterior_color` (18 dòng), `dim_interior_color` (18 dòng), `dim_fuel_type` (5 dòng), `dim_transmission` (4 dòng), `dim_drivetrain` (6 dòng), `dim_brand` (91 dòng) và `dim_grade` (612 dòng). Mỗi bảng chiều đều sử dụng tên cột tiếng Việt có dấu cho phần nội dung thuộc tính, trong khi tên bảng và tên file vẫn dùng tiếng Anh để đảm bảo tính tương thích hệ thống.
* Đáng chú ý, bảng chiều `dim_fuel_injection` có số lượng dòng rất lớn lên tới 802 dòng. Điều này phản ánh thực tế dữ liệu thô của cột `Hệ thống nạp nhiên liệu` rất hỗn loạn, chứa nhiều cách viết tắt, viết sai chính tả hoặc định dạng không đồng nhất của cùng một loại công nghệ nạp nhiên liệu.
* Việc tách các thuộc tính dạng chữ lặp đi lặp lại nhiều lần ra các bảng chiều riêng biệt giúp giảm thiểu đáng kể sự dư thừa dữ liệu, tối ưu hóa không gian lưu trữ và đảm bảo tính toàn vẹn tham chiếu. Mô hình hình sao này rất phù hợp để xây dựng mô hình dữ liệu quan hệ trong SQL hoặc import trực tiếp vào Power BI để tạo báo cáo Dashboard hiệu năng cao.

<a id="12-export-processed-and-normalized-data"></a>
## **12. Export Processed and Normalized Data**

Lưu bảng dữ liệu đã xử lý và các bảng chuẩn hóa 3NF vào thư mục `data/processed` bằng encoding `utf-8-sig`.

In [13]:
exports = {"car_detail_processed.csv": df_processed, "fact_car_listings.csv": fact_car_listings}
exports.update({f"{name}.csv": table for name, table in dimensions.items()})

export_results = []
for file_name, table in exports.items():
    path = PROCESSED_DIR / file_name
    status = "written"
    try:
        table.to_csv(path, index=False, encoding="utf-8-sig")
        check = pd.read_csv(path, low_memory=False)
    except PermissionError:
        status = "locked_existing_file"
        check = pd.read_csv(path, low_memory=False) if path.exists() else pd.DataFrame()
    export_results.append({
        "file": file_name,
        "rows": check.shape[0],
        "columns": check.shape[1],
        "exists": path.exists(),
        "status": status,
    })

export_summary = pd.DataFrame(export_results).sort_values("file").reset_index(drop=True)
display(export_summary)

,file,rows,columns,exists,status
0,car_detail_processed.csv,33848,28,True,written
1,dim_body_type.csv,10,2,True,written
2,dim_brand.csv,91,2,True,written
3,dim_condition.csv,2,2,True,written
4,dim_drivetrain.csv,6,2,True,written
5,dim_exterior_color.csv,18,2,True,written
6,dim_fuel_injection.csv,802,2,True,written
7,dim_fuel_type.csv,5,2,True,written
8,dim_grade.csv,612,3,True,written
9,dim_interior_color.csv,18,2,True,written


### **Nhận xét**

Quá trình xuất bản dữ liệu đã hoàn thành xuất sắc và ghi nhận các thông số sau:
* Tổng cộng có 13 file dữ liệu định dạng CSV đã được xuất thành công vào thư mục `data/processed/`. Tên file vẫn được giữ bằng tiếng Anh (`car_detail_processed.csv`, `fact_car_listings.csv`, `dim_origin.csv`, `dim_brand.csv`...) để đảm bảo khả năng đọc file trên các hệ thống không hỗ trợ Unicode trong tên file. Tuy nhiên, nội dung dữ liệu bên trong mỗi file (tên cột, giá trị thuộc tính) đều được giữ nguyên bằng tiếng Việt có dấu.
* Tất cả 13 file đều ghi nhận trạng thái xuất thành công (`written`) và kiểm tra sự tồn tại trên đĩa cứng đều trả về `True`.
* Các file được xuất bằng bảng mã `utf-8-sig`. Đây là một lưu ý kỹ thuật quan trọng đối với dữ liệu tiếng Việt vì `utf-8-sig` thêm ký tự BOM (Byte Order Mark) vào đầu file, giúp các ứng dụng phổ biến như Microsoft Excel có thể hiển thị chính xác các ký tự tiếng Việt có dấu trong nội dung cột (như tên hãng xe, màu sắc ngoại thất, tình trạng xe) mà không bị lỗi hiển thị font chữ khi người dùng mở trực tiếp bằng cách click đúp chuột.
Bộ dữ liệu đã xử lý và chuẩn hóa này hoàn toàn sẵn sàng cho các giai đoạn phân tích chuyên sâu tiếp theo của dự án.

**Bước tiếp theo cho Power BI:**
Import toàn bộ 13 files CSV từ `data/processed/` vào Power BI. Thiết lập các mối quan hệ 1-Nhiều từ bảng Dim hướng vào bảng Fact, và từ bảng Dim hướng vào bảng Bridge.